# Q.2 Matrix Representation

> **Requirements:** This notebook requires `sympy` (includes `sympy.physics.quantum`).
> Install with:
> ```bash
> pip install sympy
> ```

**Approach:** Expressions are simplified symbolically using the exact bosonic commutation relation $[\hat{a}, \hat{a}^\dagger] = 1$ (no truncation during algebra). Only at the final step we truncate to the qubit subspace $\{|0\rangle, |1\rangle\}$ and decompose into Pauli matrices.

In [25]:
import sympy as sp
from sympy import Rational, I, Matrix, eye, zeros, symbols, latex, simplify, kronecker_product
from sympy.physics.quantum import Dagger, Commutator
from sympy.physics.quantum.boson import BosonOp
from sympy.physics.quantum.operatorordering import normal_ordered_form
from IPython.display import display, Math
from itertools import product as iterproduct

sp.init_printing(use_latex='mathjax')

# --- Pauli basis ---
_PAULIS = {
    "I": eye(2),
    "X": Matrix([[0, 1], [1, 0]]),
    "Y": Matrix([[0, -I], [I, 0]]),
    "Z": Matrix([[1, 0], [0, -1]]),
}
_PAULI_NAMES = ["I", "X", "Y", "Z"]


def _collect_modes(expr):
    modes = set()
    if isinstance(expr, BosonOp):
        modes.add(str(expr.name))
    elif isinstance(expr, Dagger) and isinstance(expr.args[0], BosonOp):
        modes.add(str(expr.args[0].name))
    else:
        for arg in expr.args:
            modes |= _collect_modes(arg)
    return modes


class BosonExpr:
    def __init__(self, expr, modes=None):
        self.expr = expr
        self.modes = sorted(_collect_modes(expr)) if modes is None else list(modes)
        self.n = len(self.modes)
        self.dim = 2 ** self.n

    def normal_order(self):
        expanded = self.expr.expand()
        normed = normal_ordered_form(expanded, independent=True, recursive_limit=25)
        return BosonExpr(normed, self.modes)

    def truncate(self):
        normed = self.normal_order().expr
        return self._to_matrix(normed)

    def _mode_matrix(self, op_mat, mode_name):
        factors = [op_mat if m == mode_name else eye(2) for m in self.modes]
        result = factors[0]
        for f in factors[1:]:
            result = kronecker_product(result, f)
        return result

    def _to_matrix(self, expr):
        A = Matrix([[0, 1], [0, 0]])
        Ad = Matrix([[0, 0], [1, 0]])
        Id = eye(self.dim)
        if expr.is_number or expr.is_Symbol:
            return expr * Id
        if isinstance(expr, BosonOp):
            return self._mode_matrix(A if expr.is_annihilation else Ad, str(expr.name))
        if isinstance(expr, Dagger) and isinstance(expr.args[0], BosonOp):
            return self._mode_matrix(Ad, str(expr.args[0].name))
        if expr.is_Add:
            return sum((self._to_matrix(t) for t in expr.args), zeros(self.dim))
        if expr.is_Mul:
            scalar = sp.S.One
            mats = []
            for factor in expr.args:
                if factor.is_number or factor.is_Symbol:
                    scalar *= factor
                else:
                    mats.append(self._to_matrix(factor))
            result = Id
            for m in mats:
                result = result * m
            return scalar * result
        if expr.is_Pow:
            base_mat = self._to_matrix(expr.base)
            n = expr.exp
            if n.is_integer and n > 0:
                result = base_mat
                for _ in range(int(n) - 1):
                    result = result * base_mat
                return result
            if n == 0:
                return Id
            raise ValueError(f'Cannot truncate non-integer power: {expr}')
        raise TypeError(f'Cannot convert to matrix: {expr} (type {type(expr)})')

    def pauli_decompose(self):
        M = self.truncate()
        coeffs = {}
        norm = Rational(1, 2) ** self.n
        for combo in iterproduct(_PAULI_NAMES, repeat=self.n):
            factors = [_PAULIS[c] for c in combo]
            basis = factors[0]
            for f in factors[1:]:
                basis = kronecker_product(basis, f)
            c = simplify(norm * (basis * M).trace())
            if c != 0:
                coeffs[combo] = c
        return coeffs, M

    def _pauli_label(self, combo):
        pauli_tex = {"I": "I", "X": r"\sigma_x", "Y": r"\sigma_y", "Z": r"\sigma_z"}
        if self.n == 1:
            return pauli_tex[combo[0]]
        parts = []
        for name, mode in zip(combo, self.modes):
            parts.append(pauli_tex[name] + "^{" + mode + "}")
        return r" \otimes ".join(parts)

    def full_analysis(self, drop_identity=False):
        normed = self.normal_order()
        coeffs, M = self.pauli_decompose()
        display(Math(r"\textbf{Original: } " + latex(self.expr)))
        display(Math(r"\textbf{Normal-ordered: } " + latex(normed.expr)))
        dim_label = f"{self.dim}x{self.dim}"
        display(Math(r"\textbf{" + dim_label + r" matrix: } " + latex(M)))
        identity_key = tuple("I" for _ in range(self.n))
        terms = []
        for combo in sorted(coeffs.keys(), key=lambda c: [_PAULI_NAMES.index(x) for x in c]):
            if drop_identity and combo == identity_key:
                continue
            c = coeffs[combo]
            terms.append(latex(c) + r" \cdot " + self._pauli_label(combo))
        pauli_str = " + ".join(terms) if terms else "0"
        label = r"\textbf{Pauli (no const): } " if drop_identity else r"\textbf{Pauli: } "
        display(Math(label + pauli_str))
        return coeffs

In [26]:
# Define bosonic modes
a  = BosonOp("a");  ad = Dagger(a)
b  = BosonOp("b");  bd = Dagger(b)

## Verify identities 1–6

Each expression is normal-ordered symbolically using $[\hat{a}, \hat{a}^\dagger]=1$, then truncated to $2\times 2$ and decomposed into Pauli matrices.

In [28]:
# Identity 1:  a+ - a = -i*sigma_y
BosonExpr(ad - a).full_analysis(drop_identity=True);

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [29]:
# Identity 2:  a+ + a = sigma_x
BosonExpr(ad + a).full_analysis(drop_identity=True);

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [30]:
# Identity 3:  (a+ - a)^2 = -sigma_z  (dropping constant)
BosonExpr((ad - a)**2).full_analysis(drop_identity=1);

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [31]:
# Identity 4:  (a+ + a)^2 = -sigma_z  (dropping constant)
BosonExpr((ad + a)**2).full_analysis(drop_identity=True);

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [32]:
# Identity 5:  (a+ + a)^3 = 3*sigma_x
BosonExpr((ad + a)**3).full_analysis(drop_identity=True);

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [33]:
# Identity 6:  (a+ + a)^4 = -6*sigma_z  (dropping constant)
BosonExpr((ad + a)**4).full_analysis(drop_identity=True);

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Identities 7–10 (two-mode)

Mode $i = a$, mode $j = b$. Results decomposed into indexed Pauli tensor products $\sigma_\mu^a \otimes \sigma_\nu^b$.

In [34]:
# Identity 7:  (a+_i - a_i)(a+_j - a_j) = -sigma_y^i * sigma_y^j
BosonExpr((ad - a) * (bd - b)).full_analysis(drop_identity=True);

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [35]:
# Identity 8:  (a+_i + a_i)(a+_j + a_j) = sigma_x^i * sigma_x^j
BosonExpr((ad + a) * (bd + b)).full_analysis(drop_identity=True);

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [36]:
# Identity 9:  (a+_i - a_i)^3 (a+_j + a_j)
BosonExpr((ad - a)**3 * (bd + b)).full_analysis(drop_identity=True);

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [37]:
# Identity 10:  (a+_i + a_i)^2 (a+_j + a_j)^2 = sigma_z^i sigma_z^j - 2*sigma_z^i - 2*sigma_z^j
BosonExpr((ad + a)**2 * (bd + b)**2).full_analysis(drop_identity=True);

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>